# Feature Engineering and Dataset Construction

The goal of this notebook is to transform OSHA establishment-level and incident-level safety data into a model-ready dataset for workplace risk classification.

Feature engineering focuses on:
- aggregating incident-level information into establishment-level risk indicators
- creating operational and industry-related features
- reducing data leakage
- preparing scalable and reproducible inputs for machine learning pipelines

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 1. File Paths

This notebook expects EDA notebook to have saved summary_eda_clean_with_target.csv in an eda_outputs folder.

If that file is not found, this notebook will rebuild the cleaned summary dataset from the original OSHA summary ZIP.

In [2]:
BASE_DIR = Path(".")
OUTPUT_DIR = BASE_DIR / "eda_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

SUMMARY_ZIP = BASE_DIR / "ITA_300A_Summary_Data_2024_through_12-31-2025.zip"
CASE_ZIP = BASE_DIR / "ITA_Case_Detail_Data_2024_through_12-31-2025.zip"

EDA_CLEAN_FILE = OUTPUT_DIR / "summary_eda_clean_with_target.csv"

STRICT_MODEL_OUTPUT = OUTPUT_DIR / "model_dataset_strict.csv"
CASE_ENRICHED_OUTPUT = OUTPUT_DIR / "model_dataset_case_enriched.csv"

SUMMARY_ZIP.exists(), CASE_ZIP.exists(), EDA_CLEAN_FILE.exists()

(True, True, True)

## 2. Helper Functions

These functions standardize column names, clean text fields, and safely load CSV files from ZIP folders.

In [3]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )
    return df


def clean_text_columns(df):
    df = df.copy()
    text_cols = df.select_dtypes(include=["object"]).columns
    for col in text_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace("\t", " ", regex=False)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
    return df


def load_csv_from_zip(zip_path, encoding="utf-8", low_memory=False):
    with zipfile.ZipFile(zip_path) as z:
        csv_name = [name for name in z.namelist() if name.lower().endswith(".csv")][0]
        with z.open(csv_name) as f:
            df = pd.read_csv(f, encoding=encoding, low_memory=low_memory)
    df = clean_column_names(df)
    df = clean_text_columns(df)
    return df


def safe_numeric(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

## 3. Load Clean Summary Dataset

This is the establishment-level table from EDA. It should already include:

- cleaned records
- TRIR
- high-risk target variable

In [4]:
if EDA_CLEAN_FILE.exists():
    summary_clean = pd.read_csv(EDA_CLEAN_FILE)
    summary_clean = clean_column_names(summary_clean)
    summary_clean = clean_text_columns(summary_clean)

    print(f"Loaded cleaned EDA file: {summary_clean.shape}")

else:
    print("Cleaned EDA file not found. Rebuilding from original summary ZIP...")

    summary_clean = load_csv_from_zip(
        SUMMARY_ZIP,
        encoding="utf-8",
        low_memory=False
    )

    summary_clean = clean_column_names(summary_clean)
    summary_clean = clean_text_columns(summary_clean)

    numeric_cols = [
        "annual_average_employees",
        "total_hours_worked",
        "total_deaths",
        "total_dafw_cases",
        "total_djtr_cases",
        "total_other_cases",
        "total_dafw_days",
        "total_djtr_days",
        "total_injuries",
        "total_skin_disorders",
        "total_respiratory_conditions",
        "total_poisonings",
        "total_hearing_loss",
        "total_other_illnesses"
    ]

    summary_clean = safe_numeric(summary_clean, numeric_cols)

    summary_clean["total_recordable_cases"] = (
        summary_clean[
            [
                "total_deaths",
                "total_dafw_cases",
                "total_djtr_cases",
                "total_other_cases"
            ]
        ]
        .fillna(0)
        .sum(axis=1)
    )

    summary_clean["trir"] = np.where(
        summary_clean["total_hours_worked"] > 0,
        (
            summary_clean["total_recordable_cases"] * 200_000
        ) / summary_clean["total_hours_worked"],
        np.nan
    )

    summary_clean = summary_clean[
        (summary_clean["total_hours_worked"] > 0) &
        (summary_clean["annual_average_employees"] > 0) &
        (summary_clean["trir"].notna())
    ].copy()

    trir_upper = summary_clean["trir"].quantile(0.99)

    summary_clean = summary_clean[
        summary_clean["trir"] <= trir_upper
    ].copy()

    trir_threshold = summary_clean["trir"].quantile(0.75)

    summary_clean["high_risk"] = np.where(
        summary_clean["trir"] >= trir_threshold,
        1,
        0
    )

    summary_clean.to_csv(EDA_CLEAN_FILE, index=False)

    print(f"Rebuilt and saved cleaned EDA file: {summary_clean.shape}")

# ==========================================================
# FEATURE ENGINEERING ADDED OUTSIDE THE IF/ELSE
# ==========================================================

summary_clean["size_category"] = pd.cut(
    summary_clean["annual_average_employees"],
    bins=[0, 50, 250, 1000, np.inf],
    labels=["small", "medium", "large", "enterprise"]
)

summary_clean["size_category"].value_counts()

summary_clean.head()

Loaded cleaned EDA file: (392915, 38)


,id,establishment_name,establishment_id,ein,company_name,street_address,city,state,zip_code,naics_code,naics_year,industry_description,establishment_type,size,annual_average_employees,total_hours_worked,no_injuries_illnesses,total_deaths,total_dafw_cases,total_djtr_cases,total_other_cases,total_dafw_days,total_djtr_days,total_injuries,total_skin_disorders,total_respiratory_conditions,total_poisonings,total_hearing_loss,total_other_illnesses,created_timestamp,change_reason,year_filing_for,total_recordable_cases,trir,dafw_case_rate,days_away_restricted_total,avg_days_per_recordable_case,high_risk,size_category
0,2808362,AristaCare at Meadow Springs,41940,204755042.0,AristaCare at Meadow Springs LLC,845 Germantown Pike,Plymouth Meeting,PA,19462,623110,2012,Skilled nursing facilities,1.0,3,315,389754,1,0,1,5,1,1,32,7,0,0,0,0,0,28FEB25:16:35:00,<NA>,2024,7,3.592009,0.513144,33,4.714286,0,large
1,2587120,AM Braswell Jr. Food Company,41942,580667528.0,AM Braswell Jr. Food Company,226 North Zetterower Avenue,Statesboro,GA,30458,311421,2012,Canning jams and jellies,1.0,22,119,98259,1,0,7,4,0,46,69,10,0,0,0,0,1,31JAN25:20:24:00,<NA>,2024,11,22.389807,14.248059,115,10.454545,1,medium
2,2575264,"SMM Group Chesapeake, VA",41952,453807558.0,Sims Metal Management,4300 Buell Street,Chesapeake,VA,23324,423930,2012,Metal scrap and waste merchant wholesalers,1.0,22,126,278678,1,0,0,0,1,0,0,1,0,0,0,0,0,29JAN25:14:47:00,<NA>,2024,1,0.717674,0.000000,0,0.000000,0,medium
3,2575218,"SMM Group Richmond, VA",41955,453807558.0,Sims Metal Management,3220 Deepwater Terminal Road,Richmond,VA,23234,423930,2012,Metal scrap and waste merchant wholesalers,1.0,21,35,68943,2,0,0,0,0,0,0,0,0,0,0,0,0,29JAN25:14:31:00,<NA>,2024,0,0.000000,0.000000,0,NaN,0,small
4,2505064,FRP Sheet Metal Contracting Corp,41983,112459740.0,FRP Sheet Metal Contracting Corp,365 Wyandanch Ave.,West Babylon,NY,11704,238220,2012,Air-conditioning system (except window) instal...,1.0,21,25,40919,2,0,0,0,0,0,0,0,0,0,0,0,0,07JAN25:20:06:00,<NA>,2024,0,0.000000,0.000000,0,NaN,0,small


In [5]:
summary_clean.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392915 entries, 0 to 392914
Data columns (total 39 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   id                            392915 non-null  int64   
 1   establishment_name            392915 non-null  string  
 2   establishment_id              392915 non-null  int64   
 3   ein                           350008 non-null  float64 
 4   company_name                  373617 non-null  string  
 5   street_address                392914 non-null  string  
 6   city                          392914 non-null  string  
 7   state                         392915 non-null  string  
 8   zip_code                      392915 non-null  int64   
 9   naics_code                    392915 non-null  int64   
 10  naics_year                    392915 non-null  int64   
 11  industry_description          366626 non-null  string  
 12  establishment_type            

## 4. Confirm Target Variable

The target should be high_risk, where 1 means the establishment is in the top 25% of TRIR values after cleaning.

In [6]:
summary_clean["high_risk"].value_counts(dropna=False)

high_risk
0    294686
1     98229
Name: count, dtype: int64

In [7]:
summary_clean["high_risk"].value_counts(normalize=True).rename("proportion")

high_risk
0    0.749999
1    0.250001
Name: proportion, dtype: float64

In [8]:
summary_clean["size_category"] = pd.cut(
    summary_clean["annual_average_employees"],
    bins=[0, 50, 250, 1000, np.inf],
    labels=["small", "medium", "large", "enterprise"]
)

summary_clean["size_category"].value_counts()

size_category
small         217801
medium        139457
large          29700
enterprise      5957
Name: count, dtype: int64

## 5. Load Case Detail Data

The case detail dataset contains incident-level records. We will aggregate it to the establishment level so it can be merged with the summary table.


In [9]:
case_df = load_csv_from_zip(CASE_ZIP, encoding="latin1", low_memory=False)

case_numeric_cols = [
    "establishment_id", "annual_average_employees", "total_hours_worked",
    "dafw_num_away", "djtr_num_tr", "incident_outcome",
    "nature_code_pred", "part_code_pred", "event_code_pred", 
    "source_code_pred", "sec_source_code_pred"
]
case_df = safe_numeric(case_df, case_numeric_cols)

case_df.shape

(688649, 49)

In [10]:
case_df.head()

,id,establishment_id,establishment_name,ein,company_name,street_address,city,state,zip_code,naics_code,naics_year,industry_description,establishment_type,size,annual_average_employees,total_hours_worked,case_number,job_description,soc_code,soc_description,soc_reviewed,soc_probability,date_of_incident,incident_outcome,dafw_num_away,djtr_num_tr,type_of_incident,time_started_work,time_of_incident,time_unknown,date_of_death,created_timestamp,year_of_filing,new_nar_what_happened,new_nar_before_incident,new_incident_location,new_nar_injury_illness,new_nar_object_substance,new_incident_description,nature_code_pred,nature_title_pred,part_code_pred,part_title_pred,event_code_pred,event_title_pred,source_code_pred,source_title_pred,sec_source_code_pred,sec_source_title_pred
0,936394,1147869,CEVA Logistics Airtech 1680201H,232142673.0,"CEVA Contract Logistics US, Inc.",501 Airtech Parkway,Plainfield,IN,46168,493110,2022,"Bonded warehousing, general merchandise",1.0,3,337,834292,1,Warehouse Associate,53-7065,Stockers and Order Fillers,1,5.000000,16JAN2024,4,0,0,1,7:00,13:30,0.0,NaN,20JAN2025:16:52:00,2024,While cutting towards self lacerated left inde...,Opening packages using a retractable knife,Production,Laceration to left index finger,Retractable knife,Left index finger laceration from retractable ...,13.0,Surface and flesh wounds,44.0,"Hand(s), finger(s)",66.0,Contact with non-running objects or equipment,71.0,Handtools,99.0,"Other source, secondary source"
1,936437,1147869,CEVA Logistics Airtech 1680201H,232142673.0,"CEVA Contract Logistics US, Inc.",501 Airtech Parkway,Plainfield,IN,46168,493110,2022,"Bonded warehousing, general merchandise",1.0,3,337,834292,2,Warehouse Associate,53-7065,Stockers and Order Fillers,1,5.000000,17APR2024,2,1,51,1,7:00,7:30,0.0,NaN,20JAN2025:16:58:00,2024,Associate stepped backwards and tripped over a...,Scanning items on pallet,Production,Acral pain and contusion of sacrum,Wood pallet and concrete floor,acral pain Contusion of sacrum tripping over a...,13.0,Surface and flesh wounds,32.0,Exterior and musculoskeletal structures of the...,43.0,"Slip, trip, stumble or fall on same level",66.0,"Ground, travel, and support surfaces",21.0,Containers
2,936469,1147869,CEVA Logistics Airtech 1680201H,232142673.0,"CEVA Contract Logistics US, Inc.",501 Airtech Parkway,Plainfield,IN,46168,493110,2022,"Bonded warehousing, general merchandise",1.0,3,337,834292,3,Warehouse Associate,53-7065,Stockers and Order Fillers,1,5.000000,07MAY2024,3,0,27,1,7:00,9:08,0.0,NaN,20JAN2025:17:08:00,2024,Grabbed a tote with one hand bending back left...,Scanning items for put away,Production,Left wrist and hand strain,Tote,Strain of left hand and wrist after grabbing a...,14.0,Soft tissue injuries,48.0,Multiple upper extremities locations,71.0,Overexertion while moving or manipulating exte...,21.0,Containers,99.0,"Other source, secondary source"
3,1373294,801065,"Seah Steel USA LLC, P1",384017653.0,Seah Steel USA,16952 Leonard Rd.,Houston,TX,77049,331210,2012,"Pipe (e.g., heavy riveted, lock joint, seamles...",1.0,22,123,415164,1,Gag operator,51-4031,"Cutting, Punching, and Press Machine Setters, ...",0,0.999987,24AUG2024,3,0,14,1,19:00,22:50,0.0,NaN,27FEB2025:19:09:00,2024,Employee was placing a stopper on pipe and nex...,Counting Pipe and Marking,Gag Table,Right Hand Injury,Pipe,Smashed Finger,10.0,Traumatic injuries or exposures-- nonspecific ...,44.0,"Hand(s), finger(s)",66.0,Contact with non-running objects or equipment,42.0,Building materials,99.0,"Other source, secondary source"
4,1374268,801065,"Seah Steel USA LLC, P1",384017653.0,Seah Steel USA,16952 Leonard Rd.,Houston,TX,77049,331210,2012,"Pipe (e.g., heavy riveted, lock joint, seamles...",1.0,22,123,415164,2,QA operator,51-2041,Structural Metal Fabricators and Fitters,0,0.935129,15NOV2024,3,0,5,1,5:00,13:04,0.0,NaN,27FEB2025:19:16:00,2024,The employee was guided the pipe down the conv...,Getting samples for inspection.,QA Operator,Hand Injury,pipe,Smashed Finger,10.0,Trau

In [11]:
event_col = "event_title_pred"

incident_diversity = (
    case_df.groupby("establishment_id")[event_col]
    .nunique()
    .rename("incident_diversity")
)

incident_diversity.head()

establishment_id
41942    6
41952    1
41999    1
42012    1
42015    1
Name: incident_diversity, dtype: int64

In [12]:
fall_ratio = (
    case_df.assign(
        is_fall=case_df[event_col].str.contains("fall", case=False, na=False)
    )
    .groupby("establishment_id")["is_fall"]
    .mean()
    .rename("fall_incident_ratio")
)

overexertion_ratio = (
    case_df.assign(
        is_overexertion=case_df[event_col].str.contains("overexertion", case=False, na=False)
    )
    .groupby("establishment_id")["is_overexertion"]
    .mean()
    .rename("overexertion_ratio")
)

In [13]:
nature_col = "nature_title_pred"

injury_diversity = (
    case_df.groupby("establishment_id")[nature_col]
    .nunique()
    .rename("injury_diversity")
)

In [14]:
fall_ratio.head()

establishment_id
41942    0.454545
41952         0.0
41999         0.0
42012         0.0
42015         1.0
Name: fall_incident_ratio, dtype: Float64

In [15]:
overexertion_ratio.head()

establishment_id
41942    0.090909
41952         0.0
41999         0.0
42012         0.0
42015         0.0
Name: overexertion_ratio, dtype: Float64

## 6. Validate Join Key

The preferred join key is establishment_id because establishment names can contain spelling differences, tabs, abbreviations, or punctuation inconsistencies.

In [16]:
summary_ids = set(summary_clean["establishment_id"].dropna().astype(str))
case_ids = set(case_df["establishment_id"].dropna().astype(str))

overlap_ids = summary_ids.intersection(case_ids)

join_diagnostics = pd.DataFrame({
    "metric": [
        "summary_unique_establishment_ids",
        "case_unique_establishment_ids",
        "overlapping_establishment_ids",
        "pct_summary_with_case_detail",
        "pct_case_ids_in_summary"
    ],
    "value": [
        len(summary_ids),
        len(case_ids),
        len(overlap_ids),
        len(overlap_ids) / len(summary_ids) * 100 if len(summary_ids) > 0 else np.nan,
        len(overlap_ids) / len(case_ids) * 100 if len(case_ids) > 0 else np.nan
    ]
})

join_diagnostics

,metric,value
0,summary_unique_establishment_ids,392915.000000
1,case_unique_establishment_ids,64674.000000
2,overlapping_establishment_ids,64004.000000
3,pct_summary_with_case_detail,16.289528
4,pct_case_ids_in_summary,98.964035


## 7. Create Summary-Level Features

These features come from establishment-level characteristics and are safer for predictive modeling than same-year incident detail fields.

We will create:

- log employee count
- NAICS codes
- simplified industry fields
- filing year
- state indicators later during modeling

In [17]:
summary_features = summary_clean.copy()

summary_features["naics_code"] = summary_features["naics_code"].astype("string").str.extract(r"(\d+)")[0]
summary_features["naics_sector"] = summary_features["naics_code"].str[:2]

summary_features["log_employees"] = np.log1p(summary_features["annual_average_employees"])
summary_features["log_hours_worked"] = np.log1p(summary_features["total_hours_worked"])

# Optional size bins for interpretability
summary_features["employee_size_bin"] = pd.cut(
    summary_features["annual_average_employees"],
    bins=[0, 10, 50, 100, 250, 500, 1000, np.inf],
    labels=["1-10", "11-50", "51-100", "101-250", "251-500", "501-1000", "1000+"]
)

summary_features[["establishment_id", "state", "naics_code", "naics_sector", "annual_average_employees", "log_employees", "employee_size_bin", "high_risk"]].head()

,establishment_id,state,naics_code,naics_sector,annual_average_employees,log_employees,employee_size_bin,high_risk
0,41940,PA,623110,62,315,5.755742,251-500,0
1,41942,GA,311421,31,119,4.787492,101-250,1
2,41952,VA,423930,42,126,4.844187,101-250,0
3,41955,VA,423930,42,35,3.583519,11-50,0
4,41983,NY,238220,23,25,3.258097,11-50,0


## 8. Create Case-Level Flags

These flags summarize incident characteristics before aggregation.

In [18]:
case_features = case_df.copy()


for col in ["event_title_pred", "nature_title_pred", "part_title_pred", "source_title_pred", "incident_outcome", "job_description", "soc_description"]:
    if col in case_features.columns:
        case_features[col] = case_features[col].astype("string").str.lower().str.strip()

case_features["is_fall_event"] = case_features["event_title_pred"].str.contains("fall|slip|trip|stumble", case=False, na=False)
case_features["is_contact_event"] = case_features["event_title_pred"].str.contains("contact|struck|caught", case=False, na=False)
case_features["is_overexertion_event"] = case_features["event_title_pred"].str.contains("overexertion|bodily reaction", case=False, na=False)
case_features["is_transportation_event"] = case_features["event_title_pred"].str.contains("transportation|vehicle", case=False, na=False)
case_features["is_equipment_source"] = case_features["source_title_pred"].str.contains("machine|machinery|equipment|tool", case=False, na=False)
case_features["is_surface_wound"] = case_features["nature_title_pred"].str.contains("surface|wound|cut|laceration", case=False, na=False)
case_features["is_musculoskeletal"] = case_features["nature_title_pred"].str.contains("sprain|strain|musculoskeletal|pain", case=False, na=False)


case_features["dafw_num_away"] = pd.to_numeric(case_features.get("dafw_num_away"), errors="coerce").fillna(0)
case_features["djtr_num_tr"] = pd.to_numeric(case_features.get("djtr_num_tr"), errors="coerce").fillna(0)
case_features["is_severe_case"] = (
    (case_features["dafw_num_away"] > 0) |
    (case_features["djtr_num_tr"] > 0) |
    (case_features.get("date_of_death", pd.Series(index=case_features.index)).notna())
)

case_features[["establishment_id", "event_title_pred", "nature_title_pred", "is_fall_event", "is_contact_event", "is_severe_case"]].head()

,establishment_id,event_title_pred,nature_title_pred,is_fall_event,is_contact_event,is_severe_case
0,1147869,contact with non-running objects or equipment,surface and flesh wounds,False,True,False
1,1147869,"slip, trip, stumble or fall on same level",surface and flesh wounds,True,False,True
2,1147869,overexertion while moving or manipulating exte...,soft tissue injuries,False,False,True
3,801065,contact with non-running objects or equipment,traumatic injuries or exposures-- nonspecific ...,False,True,True
4,801065,contact with non-running objects or equipment,traumatic injuries or exposures-- nonspecific ...,False,True,True


## 9. Aggregate Case Detail to Establishment Level

This creates one row per establishment from the case-detail dataset.

In [19]:
case_agg = (
    case_features
    .groupby("establishment_id")
    .agg(
        case_detail_count=("id", "count"),
        unique_case_numbers=("case_number", "nunique"),
        unique_job_descriptions=("job_description", "nunique"),
        unique_soc_descriptions=("soc_description", "nunique"),
        event_type_diversity=("event_title_pred", "nunique"),
        nature_type_diversity=("nature_title_pred", "nunique"),
        body_part_diversity=("part_title_pred", "nunique"),
        source_type_diversity=("source_title_pred", "nunique"),
        pct_fall_events=("is_fall_event", "mean"),
        pct_contact_events=("is_contact_event", "mean"),
        pct_overexertion_events=("is_overexertion_event", "mean"),
        pct_transportation_events=("is_transportation_event", "mean"),
        pct_equipment_source=("is_equipment_source", "mean"),
        pct_surface_wounds=("is_surface_wound", "mean"),
        pct_musculoskeletal=("is_musculoskeletal", "mean"),
        severe_case_ratio=("is_severe_case", "mean"),
        total_days_away_detail=("dafw_num_away", "sum"),
        total_transfer_restriction_days_detail=("djtr_num_tr", "sum")
    )
    .reset_index()
)

case_agg.head()

,establishment_id,case_detail_count,unique_case_numbers,unique_job_descriptions,unique_soc_descriptions,event_type_diversity,nature_type_diversity,body_part_diversity,source_type_diversity,pct_fall_events,pct_contact_events,pct_overexertion_events,pct_transportation_events,pct_equipment_source,pct_surface_wounds,pct_musculoskeletal,severe_case_ratio,total_days_away_detail,total_transfer_restriction_days_detail
0,41942,11,11,7,3,6,5,7,8,0.454545,0.363636,0.090909,0.0,0.181818,0.363636,0.0,1.0,41,69
1,41952,1,1,1,1,1,1,1,1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
2,41999,1,1,1,1,1,1,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,3,12
3,42012,1,1,1,1,1,1,1,1,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0,0
4,42015,1,1,1,1,1,1,1,1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0


In [20]:
case_agg.describe().T

,count,mean,std,min,25%,50%,75%,max
establishment_id,64674.0,1005368.447568,369152.446058,41942.0,824424.25,1115721.0,1305334.25,1422813.0
case_detail_count,64674.0,10.648004,31.566808,1.0,2.0,5.0,10.0,2879.0
unique_case_numbers,64674.0,10.64788,31.56683,1.0,2.0,5.0,10.0,2879.0
unique_job_descriptions,64674.0,4.424514,6.53677,0.0,1.0,3.0,5.0,270.0
unique_soc_descriptions,64674.0,2.922055,2.785782,1.0,1.0,2.0,4.0,53.0
event_type_diversity,64674.0,3.539367,2.435094,0.0,2.0,3.0,5.0,21.0
nature_type_diversity,64674.0,2.764666,1.688973,0.0,1.0,2.0,4.0,20.0
body_part_diversity,64674.0,3.224433,2.639726,0.0,1.0,3.0,4.0,17.0
source_type_diversity,64674.0,4.16158,3.425336,0.0,2.0,3.0,6.0,46.0
pct_fall_events,64674.0,0.268585,0.278203,0.0,0.0,0.212121,0.4,1.0


## 10. Merge Summary Features with Case Aggregates

This creates the case-enriched establishment dataset. Establishments with no case-detail records receive zeros for aggregated incident-detail features.


In [21]:
case_enriched = summary_features.merge(
    case_agg,
    on="establishment_id",
    how="left",
    validate="one_to_one"
)

case_feature_cols = case_agg.columns.drop("establishment_id")
case_enriched[case_feature_cols] = case_enriched[case_feature_cols].fillna(0)

case_enriched.shape

(392915, 61)

In [22]:
summary_clean = summary_clean.merge(
    incident_diversity,
    on="establishment_id",
    how="left"
)

summary_clean = summary_clean.merge(
    fall_ratio,
    on="establishment_id",
    how="left"
)

summary_clean = summary_clean.merge(
    overexertion_ratio,
    on="establishment_id",
    how="left"
)

In [23]:
case_enriched[["establishment_id", "state", "naics_sector", "high_risk", "case_detail_count", "event_type_diversity", "pct_fall_events", "severe_case_ratio"]].head()

,establishment_id,state,naics_sector,high_risk,case_detail_count,event_type_diversity,pct_fall_events,severe_case_ratio
0,41940,PA,62,0,0.0,0.0,0.0,0.0
1,41942,GA,31,1,11.0,6.0,0.454545,1.0
2,41952,VA,42,0,1.0,1.0,0.0,0.0
3,41955,VA,42,0,0.0,0.0,0.0,0.0
4,41983,NY,23,0,0.0,0.0,0.0,0.0


In [24]:
summary_clean["hours_per_employee"] = (
    summary_clean["total_hours_worked"] /
    summary_clean["annual_average_employees"]
)

## 11. Leakage Review

Because the target is based on current-year TRIR, features that directly describe current-year injury totals should not be used in the strict predictive model.

### Exclude from strict modeling:

- `trir`
- `total_recordable_cases`
- `total_deaths`
- `total_dafw_cases`
- `total_djtr_cases`
- `total_other_cases`
- `total_dafw_days`
- `total_djtr_days`
- other current-year injury/illness totals
- case-detail count and event features, unless the project is framed as risk profiling instead of prospective prediction

The strict model dataset below keeps basic establishment characteristics and the high-risk label.

In [25]:
leakage_cols = [
    "trir",
    "total_recordable_cases",
    "total_deaths",
    "total_dafw_cases",
    "total_djtr_cases",
    "total_other_cases",
    "total_dafw_days",
    "total_djtr_days",
    "total_injuries",
    "total_skin_disorders",
    "total_respiratory_conditions",
    "total_poisonings",
    "total_hearing_loss",
    "total_other_illnesses",
    "no_injuries_illnesses",
    "dafw_case_rate",
    "days_away_restricted_total",
    "avg_days_per_recordable_case",
    "total_hours_worked"
]

admin_cols = [
    "id", "ein", "establishment_name", "company_name", "street_address",
    "city", "zip_code", "created_timestamp", "change_reason"
]

case_detail_cols = list(case_feature_cols)

exclude_strict = set(leakage_cols + admin_cols + case_detail_cols)

strict_model_df = case_enriched.drop(columns=[col for col in exclude_strict if col in case_enriched.columns]).copy()

strict_model_df.shape

(392915, 15)

In [26]:
strict_model_df.head()

,establishment_id,state,naics_code,naics_year,industry_description,establishment_type,size,annual_average_employees,year_filing_for,high_risk,size_category,naics_sector,log_employees,log_hours_worked,employee_size_bin
0,41940,PA,623110,2012,Skilled nursing facilities,1.0,3,315,2024,0,large,62,5.755742,12.873274,251-500
1,41942,GA,311421,2012,Canning jams and jellies,1.0,22,119,2024,1,medium,31,4.787492,11.495372,101-250
2,41952,VA,423930,2012,Metal scrap and waste merchant wholesalers,1.0,22,126,2024,0,medium,42,4.844187,12.537816,101-250
3,41955,VA,423930,2012,Metal scrap and waste merchant wholesalers,1.0,21,35,2024,0,small,42,3.583519,11.141050,11-50
4,41983,NY,238220,2012,Air-conditioning system (except window) instal...,1.0,21,25,2024,0,small,23,3.258097,10.619374,11-50


## 12. Basic Missing Value Handling

For dataset construction, keep this simple. More advanced imputation can happen inside the modeling pipeline.

In [27]:
assert "high_risk" in strict_model_df.columns

missing_summary = (
    strict_model_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_pct")
    .to_frame()
)

missing_summary.head(20)

,missing_pct
industry_description,0.066908
establishment_type,0.001400
establishment_id,0.000000
naics_code,0.000000
state,0.000000
naics_year,0.000000
size,0.000000
annual_average_employees,0.000000
year_filing_for,0.000000
high_risk,0.000000


In [28]:
high_missing_cols = missing_summary[missing_summary["missing_pct"] > 0.50].index.tolist()
high_missing_cols = [col for col in high_missing_cols if col != "high_risk"]

strict_model_df = strict_model_df.drop(columns=high_missing_cols)

for col in strict_model_df.columns:
    if pd.api.types.is_numeric_dtype(strict_model_df[col]):
        strict_model_df[col] = strict_model_df[col].fillna(strict_model_df[col].median())
    else:
        strict_model_df[col] = strict_model_df[col].astype("object").fillna("Unknown")

strict_model_df.isna().sum().sum()

np.int64(0)

## 13. Final Dataset Checks

Before saving, confirm:

- one row per establishment
- target exists
- target distribution is reasonable
- no obvious leakage columns remain in strict dataset

In [29]:
final_checks = {
    "rows": strict_model_df.shape[0],
    "columns": strict_model_df.shape[1],
    "duplicate_establishment_ids": strict_model_df["establishment_id"].duplicated().sum() if "establishment_id" in strict_model_df.columns else "not included",
    "target_positive_rate": strict_model_df["high_risk"].mean(),
}

final_checks

{'rows': 392915,
 'columns': 15,
 'duplicate_establishment_ids': np.int64(0),
 'target_positive_rate': np.float64(0.25000063626993113)}

In [30]:
strict_model_df["high_risk"].value_counts(normalize=True)

high_risk
0    0.749999
1    0.250001
Name: proportion, dtype: float64

In [31]:
remaining_leakage_cols = [col for col in leakage_cols if col in strict_model_df.columns]
remaining_leakage_cols

[]

## 14. Save Final Datasets

The strict dataset is the recommended input for the modeling notebook.

The case-enriched dataset is saved separately for optional business profiling and additional analysis.

In [32]:
strict_model_df.to_csv(STRICT_MODEL_OUTPUT, index=False)
case_enriched.to_csv(CASE_ENRICHED_OUTPUT, index=False)

print(f"Saved strict modeling dataset to: {STRICT_MODEL_OUTPUT}")
print(f"Saved case-enriched dataset to: {CASE_ENRICHED_OUTPUT}")
print("Strict dataset shape:", strict_model_df.shape)
print("Case-enriched dataset shape:", case_enriched.shape)

Saved strict modeling dataset to: eda_outputs\model_dataset_strict.csv
Saved case-enriched dataset to: eda_outputs\model_dataset_case_enriched.csv
Strict dataset shape: (392915, 15)
Case-enriched dataset shape: (392915, 61)


# Feature Engineering Summary

This notebook created a model-ready establishment-level dataset for high-risk workplace safety classification.

Key decisions:

- Used the cleaned EDA dataset as the base table.
- Preserved one row per establishment.
- Created business-relevant establishment features such as NAICS sector, employee-size bins, and log employee count.
- Aggregated case-detail records into establishment-level incident profile features.
- Created two datasets: a strict predictive modeling dataset and a case-enriched risk-profiling dataset.
- Removed direct leakage variables from the strict modeling dataset.


## Transition to Modeling

The final modeling dataset includes engineered operational, industry, and incident-pattern features suitable for machine learning classification workflows.

The next phase will evaluate multiple classification algorithms while incorporating preprocessing pipelines, regularization techniques, class imbalance handling, and model explainability methods.